In [1]:
import geoai
import leafmap.foliumap as leafmap

In [8]:
sifnos_bbox = [24.60, 36.93, 24.76, 37.05]

In [9]:
items = geoai.pc_stac_search(
    collection="sentinel-2-l2a",
    bbox=sifnos_bbox,
    time_range="2026-06-01/2026-09-30",
    query={"eo:cloud_cover": {"lt": 10}},
    max_items=50,
)
print(f"Found {len(items)} scenes")

Found 50 scenes


In [11]:
from shapely.geometry import box

sifnos_geom = box(*sifnos_bbox)  # your [min_lon, min_lat, max_lon, max_lat] from earlier

best_item = None
best_overlap = 0

for item in items:
    item_geom = box(*item.bbox)
    overlap = sifnos_geom.intersection(item_geom).area
    if overlap > best_overlap:
        best_overlap = overlap
        best_item = item

coverage_pct = (best_overlap / sifnos_geom.area) * 100
print(f"Best match: {best_item.id}, covers {coverage_pct:.1f}% of Sifnos's bounding box")

Best match: S2A_MSIL2A_20260928T091331_R050_T35SKB_20260928T142309, covers 100.0% of Sifnos's bounding box


In [13]:
import planetary_computer

signed_item = planetary_computer.sign(best_item)
print(signed_item.assets["visual"].href)

https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/35/S/KB/2026/09/28/S2A_MSIL2A_20260928T091331_N0513_R050_T35SKB_20260928T142309.SAFE/GRANULE/L2A_T35SKB_A058849_20260928T091325/IMG_DATA/R10m/T35SKB_20260928T091331_TCI_10m.tif?st=2026-10-03T13%3A17%3A00Z&se=2026-10-04T14%3A02%3A00Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-10-04T12%3A50%3A15Z&ske=2026-10-11T12%3A50%3A15Z&sks=b&skv=2025-07-05&sig=x68/a7YwskCz9i1wF54i5zH93lRc/6hiwOe0rsfl6rU%3D


In [14]:
m = geoai.view_pc_item(item=signed_item, assets=["visual"])
m.zoom_to_bounds(sifnos_bbox)

from IPython.display import display
display(m)